<a href="https://colab.research.google.com/github/VaibhavBhujbal/end-to-end-rag/blob/main/end_to_end_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End RAG Pipeline: Chunking → Embeddings → Retrieval → Reranking → Hallucination Evaluation

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** system from scratch, with every design decision explained. It runs entirely on the **free Colab CPU tier** (a GPU runtime makes it faster but is not required).

## Why RAG?

LLMs have two structural weaknesses:
1. **Knowledge cutoff & missing private data** — the model has never seen your internal documents.
2. **Hallucination** — when the model doesn't know, it often fabricates fluent, confident nonsense.

RAG addresses both by **retrieving relevant text at query time** and instructing the LLM to answer *only from that text*. The quality of a RAG system is therefore dominated by the quality of retrieval — which is why most of this notebook is about retrieval, not generation.

## The pipeline we will build

```
                    ┌────────────  OFFLINE (indexing)  ────────────┐
 Documents ──► Chunking ──► Embedding ──► Vector Index (FAISS)
                                     └──► Lexical Index (BM25)
                    └───────────────────────────────────────────────┘

                    ┌────────────  ONLINE (query time)  ────────────┐
 Query ──► (optional) Query Expansion / HyDE
       ──► Dense Retrieval ─┐
       ──► BM25 Retrieval ──┼──► Hybrid Fusion (RRF) ──► Cross-Encoder Reranking
                            │
       ──► top-k context ──► LLM Generation ──► Hallucination / Faithfulness Evaluation
                    └───────────────────────────────────────────────┘
```

## What each section covers

| # | Section | Key concepts |
|---|---------|--------------|
| 1 | Setup | Library roles |
| 2 | Corpus + eval set | Ground truth for measuring retrieval |
| 3 | **Chunking strategies** | Fixed, recursive, sentence, semantic |
| 4 | **Embeddings** | Bi-encoders, normalization, cosine vs dot product |
| 5 | Vector index | FAISS, exact vs approximate search |
| 6 | Lexical retrieval | BM25, when keywords beat vectors |
| 7 | **Hybrid retrieval** | Reciprocal Rank Fusion |
| 8 | **Retrieval optimisation** | Multi-query expansion, HyDE, MMR |
| 9 | **Reranking** | Cross-encoders, two-stage retrieval |
| 10 | Retrieval evaluation | Hit@k, MRR — comparing all strategies |
| 11 | Generation | Grounded prompting |
| 12 | **Hallucination evaluation** | NLI faithfulness, answer relevance, LLM-as-judge |
| 13 | End-to-end pipeline class | Putting it all together |


---
# 1. Setup — installing the toolchain

**What this cell does:** installs every library the notebook needs. Understanding *why* each one is here matters more than the install command itself:

| Library | Role in the pipeline |
|---|---|
| `sentence-transformers` | Loads **bi-encoder** embedding models (for dense retrieval) *and* **cross-encoder** models (for reranking and NLI-based faithfulness checks). One library, three jobs. |
| `faiss-cpu` | Facebook AI Similarity Search — the vector index. Stores chunk embeddings and answers "which vectors are closest to this query vector?" in milliseconds. |
| `rank_bm25` | A pure-Python BM25 implementation — the classic **lexical** (keyword) retrieval algorithm that still beats embeddings on exact-term matches (IDs, acronyms, product codes). |
| `transformers` + `accelerate` | Hugging Face runtime for the generator LLM (`flan-t5-base`) so the notebook needs **no API key**. In production you would swap this for Claude/GPT via API — the pipeline logic is identical. |
| `nltk` | Sentence tokenizer used by the sentence-based and semantic chunkers. |

**Why pin nothing?** For a teaching notebook, latest stable versions are fine. In production you would pin exact versions (`faiss-cpu==1.8.0`) for reproducibility.

> ⏱️ Takes ~1–2 minutes on Colab. The `-q` flag suppresses noisy output.

In [ ]:
# Install all dependencies (quietly)
!pip install -q sentence-transformers faiss-cpu rank_bm25 transformers accelerate nltk

# Download the NLTK sentence tokenizer models.
# 'punkt' is the classic model; newer NLTK versions also need 'punkt_tab'.
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.9 MB/s eta 0:00:00
✅ Setup complete


**What this cell does:** imports everything and fixes random seeds.

- `numpy` — all similarity math is vector/matrix operations.
- `SentenceTransformer` vs `CrossEncoder` — the two fundamentally different scoring architectures we'll contrast in Section 9.
- Setting seeds (`random.seed`, `np.random.seed`, `torch.manual_seed`) makes runs **reproducible** — essential when you're comparing retrieval strategies and need to know a metric moved because of your change, not randomness.

In [ ]:
import random
import re
import numpy as np
import torch
import pandas as pd

from sentence_transformers import SentenceTransformer, CrossEncoder
from nltk.tokenize import sent_tokenize

# Reproducibility: identical results on every run
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}")

Running on: cpu


---
# 2. The corpus and evaluation set

**What this cell does:** creates a small knowledge base of 8 documents about ML/data topics, plus an **evaluation set** of questions where we *know* which document contains the answer.

**Why a synthetic corpus?**
- Small enough to inspect by eye — you can verify every retrieval result yourself.
- Deliberately contains **near-duplicate topics** (two documents both mention "embeddings", two mention "evaluation") so retrieval is *not* trivially easy — the retriever has to discriminate, just like in real enterprise corpora.

**Why the eval set matters (the most skipped step in real projects):**
Every question is labelled with `relevant_doc` — the ID of the document that contains the answer. This ground truth lets us compute **Hit@k** and **MRR** in Section 10 and *prove* which retrieval strategy is best, instead of eyeballing a few queries. Without labelled queries, RAG tuning is guesswork.

Note the last two questions are intentionally hard:
- `q7` uses **paraphrased vocabulary** ("catch made-up facts" instead of "hallucination detection") — this is where lexical BM25 struggles and dense embeddings shine.
- `q8` uses an **exact rare keyword** ("RRF") — this is where BM25 shines and embeddings can be fuzzy.

In [ ]:
# ---- Knowledge base: 8 documents -------------------------------------------
DOCUMENTS = {
    "doc_chunking": """Chunking is the process of splitting documents into smaller pieces before
indexing them in a retrieval system. Chunk size involves a fundamental trade-off: small chunks
produce precise, focused embeddings but lose surrounding context, while large chunks preserve
context but dilute the embedding signal, because a single vector must summarise many topics.
A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent.
Overlap ensures that a sentence falling on a chunk boundary still appears intact in at least
one chunk. Recursive character splitting tries to break text at natural boundaries, first at
paragraphs, then sentences, then words, only cutting mid-word as a last resort.""",

    "doc_embeddings": """Text embeddings map sentences or passages to dense numeric vectors such
that semantically similar texts are close together in vector space. Bi-encoder models such as
all-MiniLM-L6-v2 encode the query and the document independently into 384-dimensional vectors,
which makes them fast: document vectors are precomputed offline and only the query is embedded
at search time. Similarity is measured with cosine similarity or, if vectors are normalised to
unit length, with a simple dot product, since the two are then mathematically equivalent.
Embedding models are trained with contrastive objectives that pull matching pairs together and
push non-matching pairs apart.""",

    "doc_vectordb": """A vector database stores embeddings and answers nearest-neighbour queries.
FAISS offers exact search with IndexFlatIP, which compares the query against every stored vector
and guarantees perfect recall, but scales linearly with corpus size. For millions of vectors,
approximate nearest neighbour indexes are used instead: IVF partitions the space into clusters
and searches only the closest clusters, while HNSW builds a navigable small-world graph that
achieves sub-millisecond search with a small recall loss. The choice is a recall versus latency
versus memory trade-off.""",

    "doc_bm25": """BM25 is a lexical ranking function based on term frequency and inverse document
frequency. It rewards documents that contain the exact query terms, with diminishing returns for
repeated terms and a penalty for very long documents. BM25 requires no training and excels at
exact keyword matches such as error codes, acronyms, product identifiers, and rare proper nouns,
which dense embedding models often blur together. Its weakness is vocabulary mismatch: it cannot
match synonyms or paraphrases, because it has no notion of meaning beyond the surface tokens.""",

    "doc_hybrid": """Hybrid retrieval combines dense vector search with lexical BM25 search to get
the best of both worlds. A popular fusion method is Reciprocal Rank Fusion, abbreviated RRF, which
scores each document as the sum of 1 / (k + rank) across the ranked lists it appears in, with k
typically set to 60. RRF needs no score normalisation because it uses only ranks, not raw scores,
which makes it robust when the two retrievers produce scores on completely different scales.
Hybrid retrieval consistently outperforms either method alone on heterogeneous real-world queries.""",

    "doc_rerank": """Reranking is a second-stage refinement applied after initial retrieval.
A cross-encoder feeds the query and a candidate passage through the transformer together, so
attention flows between every query token and every passage token, producing a far more accurate
relevance score than a bi-encoder. The cost is speed: the cross-encoder must run once per
query-passage pair at query time, so it is only applied to a shortlist, for example the top 20
candidates from fast first-stage retrieval, from which it selects the final top 3 to 5 passages.
This two-stage design, fast recall then accurate precision, is standard in production search.""",

    "doc_halluc": """Hallucination in retrieval-augmented generation means the model produces
statements that are not supported by the retrieved context. Faithfulness evaluation checks whether
every claim in the generated answer is entailed by the retrieved passages. One automatic approach
uses a natural language inference model: the answer is split into sentences, and each sentence is
checked for entailment against the context, yielding a faithfulness score equal to the fraction of
supported sentences. Another approach is LLM-as-judge, where a strong model is prompted to verify
each claim against the context. Frameworks such as RAGAS package these metrics, including
faithfulness, answer relevance, and context precision.""",

    "doc_eval": """Retrieval quality is measured with rank-based metrics on a labelled query set.
Hit rate at k, also called recall at k, is the fraction of queries for which a relevant document
appears anywhere in the top k results. Mean reciprocal rank, MRR, averages 1 divided by the rank
of the first relevant document, so it rewards putting the right document at position one. A solid
evaluation set contains queries with known relevant documents, including paraphrased queries that
avoid the document's own vocabulary, which specifically stress-tests semantic matching.""",
}

# ---- Labelled evaluation queries --------------------------------------------
EVAL_QUERIES = [
    {"q": "What chunk size should I use and why does overlap help?",          "relevant_doc": "doc_chunking"},
    {"q": "How do bi-encoders compute sentence similarity?",                  "relevant_doc": "doc_embeddings"},
    {"q": "When should I use HNSW instead of exact search in FAISS?",         "relevant_doc": "doc_vectordb"},
    {"q": "Why does keyword search work better for error codes?",             "relevant_doc": "doc_bm25"},
    {"q": "How does Reciprocal Rank Fusion combine two result lists?",        "relevant_doc": "doc_hybrid"},
    {"q": "Why are cross-encoders slower but more accurate?",                 "relevant_doc": "doc_rerank"},
    {"q": "How can I automatically catch made-up facts in generated answers?","relevant_doc": "doc_halluc"},   # paraphrase — hard for BM25
    {"q": "What is RRF?",                                                     "relevant_doc": "doc_hybrid"},   # rare acronym — easy for BM25
    {"q": "What does MRR measure in search evaluation?",                      "relevant_doc": "doc_eval"},
]

total_words = sum(len(d.split()) for d in DOCUMENTS.values())
print(f"{len(DOCUMENTS)} documents, {total_words} words total, {len(EVAL_QUERIES)} labelled eval queries")

8 documents, 755 words total, 9 labelled eval queries


---
# 3. Chunking strategies

Chunking is the **highest-leverage, cheapest-to-change** knob in a RAG system. The chunk is the *unit of retrieval*: if the answer is split across two chunks, or buried in a chunk full of unrelated text, no amount of clever retrieval downstream can fully recover.

We implement four strategies from scratch (no LangChain, so you see exactly what happens):

| Strategy | How it splits | Pros | Cons |
|---|---|---|---|
| **Fixed-size** | Every N characters, with overlap | Trivial, predictable size | Cuts mid-sentence, destroys meaning at boundaries |
| **Recursive** | Paragraph → sentence → word, falling back only when needed | Respects natural structure, bounded size | Slightly more complex |
| **Sentence-based** | Groups of whole sentences | Never breaks a sentence | Chunk sizes vary |
| **Semantic** | Splits where **embedding similarity between adjacent sentences drops** | Boundaries follow *topic shifts* | Requires running the embedding model at index time (cost) |

**The core trade-off to internalise:**
- **Small chunks** → each embedding represents one focused idea → *precise retrieval*, but the LLM may receive too little surrounding context to answer.
- **Large chunks** → rich context for the LLM, but the embedding becomes a blurry average of several topics → *worse retrieval*.
- **Overlap** is insurance: a fact near a boundary appears intact in at least one chunk.

### 3.1 Fixed-size chunking with overlap

**What this cell does:** implements the simplest possible chunker — slide a window of `chunk_size` characters, stepping `chunk_size - overlap` each time.

Walk through the logic:
1. `step = chunk_size - overlap` — how far the window advances. With size 400 / overlap 80, consecutive chunks share 80 characters.
2. The `while` loop slices `text[start : start + chunk_size]` until the text is exhausted.
3. Each chunk keeps metadata (`doc_id`, `chunk_id`) — **always carry metadata**; at answer time you need to cite *where* the context came from.

Look at the printed output: the chunk boundary cuts **mid-sentence, even mid-word** — this is precisely the weakness the other strategies fix.

In [ ]:
def fixed_size_chunk(text, doc_id, chunk_size=400, overlap=80):
    """Split `text` into overlapping windows of `chunk_size` characters."""
    text = re.sub(r"\s+", " ", text).strip()   # normalise whitespace
    step = chunk_size - overlap                  # window advance per iteration
    chunks, start, i = [], 0, 0
    while start < len(text):
        piece = text[start : start + chunk_size]
        chunks.append({"doc_id": doc_id, "chunk_id": f"{doc_id}_fixed_{i}", "text": piece})
        start += step
        i += 1
    return chunks

# Demo on one document
demo = fixed_size_chunk(DOCUMENTS["doc_chunking"], "doc_chunking")
print(f"Produced {len(demo)} chunks\n")
print("--- end of chunk 0 ---")
print("...", demo[0]["text"][-60:])
print("\n--- start of chunk 1 (note the 80-char overlap) ---")
print(demo[1]["text"][:60], "...")

Produced 3 chunks

--- end of chunk 0 ---
... se many topics. A common starting point is 200 to 500 tokens

--- start of chunk 1 (note the 80-char overlap) ---
 vector must summarise many topics. A common starting point  ...


### 3.2 Recursive character chunking

**What this cell does:** implements the strategy popularised by LangChain's `RecursiveCharacterTextSplitter`, from scratch.

The idea is a **priority list of separators**: `["\n\n", ". ", " "]` (paragraph → sentence → word).
1. If the text already fits in `chunk_size`, return it as-is.
2. Otherwise split on the **first (most natural) separator**, then greedily pack the resulting pieces into chunks up to `chunk_size`.
3. If a single piece is *still* too big (e.g. one giant paragraph), **recurse** on it with the *next* separator in the list.

The result: chunks respect the most natural boundary possible, and only degrade to word-level splits when forced. This is the sensible **default chunker for most production systems**.

In [ ]:
def recursive_chunk(text, doc_id, chunk_size=400, separators=("\n\n", ". ", " "), _depth=0):
    """Split on the most natural separator first; recurse with finer separators only if needed."""
    text = text.strip()
    if len(text) <= chunk_size:                      # base case: already fits
        return [text] if text else []

    if _depth >= len(separators):                    # no separators left: hard cut
        return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    sep = separators[_depth]
    parts = [p + (sep if sep != " " else " ") for p in text.split(sep) if p.strip()]

    pieces, buf = [], ""
    for part in parts:
        if len(part) > chunk_size:                   # a single part is too big → recurse deeper
            if buf:
                pieces.append(buf); buf = ""
            pieces.extend(recursive_chunk(part, doc_id, chunk_size, separators, _depth + 1))
        elif len(buf) + len(part) <= chunk_size:     # greedily pack into current buffer
            buf += part
        else:                                        # buffer full → flush, start new one
            pieces.append(buf); buf = part
    if buf:
        pieces.append(buf)

    if _depth == 0:  # attach metadata only at the top level
        return [{"doc_id": doc_id, "chunk_id": f"{doc_id}_rec_{i}", "text": p.strip()}
                for i, p in enumerate(pieces)]
    return pieces

demo = recursive_chunk(DOCUMENTS["doc_chunking"], "doc_chunking")
for c in demo:
    print(f"[{len(c['text']):>3} chars] {c['text'][:80]}...")

[110 chars] Chunking is the process of splitting documents into smaller pieces before
indexi...
[396 chars] Chunk size involves a fundamental trade-off: small chunks
produce precise, focus...
[ 43 chars] still appears intact in at least
one chunk....
[162 chars] Recursive character splitting tries to break text at natural boundaries, first a...


### 3.3 Sentence-based chunking

**What this cell does:** uses NLTK's `sent_tokenize` to split into sentences, then packs `sentences_per_chunk` whole sentences into each chunk, overlapping by `overlap_sentences`.

**Why sentences?** A sentence is the smallest unit that carries a complete claim. Guaranteeing chunks are built of whole sentences means:
- embeddings never encode a broken half-thought,
- the faithfulness checker in Section 12 (which operates sentence-by-sentence) aligns perfectly with the chunks.

The trade-off is variable chunk length — a chunk of three long sentences may be 3× a chunk of three short ones.

In [ ]:
def sentence_chunk(text, doc_id, sentences_per_chunk=3, overlap_sentences=1):
    """Pack whole sentences into chunks; consecutive chunks share `overlap_sentences`."""
    text = re.sub(r"\s+", " ", text).strip()
    sents = sent_tokenize(text)
    step = sentences_per_chunk - overlap_sentences
    chunks = []
    for i, start in enumerate(range(0, len(sents), step)):
        group = sents[start : start + sentences_per_chunk]
        if not group:
            break
        chunks.append({"doc_id": doc_id, "chunk_id": f"{doc_id}_sent_{i}",
                       "text": " ".join(group)})
        if start + sentences_per_chunk >= len(sents):
            break
    return chunks

demo = sentence_chunk(DOCUMENTS["doc_chunking"], "doc_chunking")
for c in demo:
    print(f"[{len(c['text']):>3} chars] {c['text'][:80]}...")

[447 chars] Chunking is the process of splitting documents into smaller pieces before indexi...
[357 chars] A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to ...


### 3.4 Semantic chunking

**What this cell does:** places chunk boundaries where the **topic shifts**, detected via embeddings:

1. Embed every sentence individually with a small bi-encoder (`all-MiniLM-L6-v2` — we'll meet it properly in Section 4).
2. Compute cosine similarity between each **adjacent pair** of sentences.
3. Any similarity below a percentile threshold (here the 30th percentile of all adjacent similarities) is declared a **breakpoint** — the topic changed.
4. Sentences between breakpoints form a chunk.

**When is this worth the extra cost?** Long documents that weave through multiple topics (meeting transcripts, reports, policy documents). For short, single-topic documents (like ours), recursive chunking is nearly as good and much cheaper — semantic chunking runs the embedding model over *every sentence* at index time.

Note we load the embedding model here for the first time — it's ~80 MB and downloads once.

In [ ]:
# Load the bi-encoder once; it's reused throughout the notebook.
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

def semantic_chunk(text, doc_id, percentile=30):
    """Break chunks where adjacent-sentence cosine similarity dips below a percentile threshold."""
    text = re.sub(r"\s+", " ", text).strip()
    sents = sent_tokenize(text)
    if len(sents) <= 2:
        return [{"doc_id": doc_id, "chunk_id": f"{doc_id}_sem_0", "text": text}]

    # 1) embed every sentence (normalized → dot product == cosine similarity)
    emb = embedder.encode(sents, normalize_embeddings=True)

    # 2) similarity between each adjacent sentence pair
    sims = np.array([float(np.dot(emb[i], emb[i+1])) for i in range(len(sents)-1)])

    # 3) breakpoints = dips below the chosen percentile
    threshold  = np.percentile(sims, percentile)
    breakpoints = [i for i, s in enumerate(sims) if s < threshold]

    # 4) assemble chunks between breakpoints
    chunks, start = [], 0
    for i, bp in enumerate(breakpoints + [len(sents) - 1]):
        chunks.append({"doc_id": doc_id, "chunk_id": f"{doc_id}_sem_{i}",
                       "text": " ".join(sents[start : bp + 1])})
        start = bp + 1
    return [c for c in chunks if c["text"]]

demo = semantic_chunk(DOCUMENTS["doc_chunking"], "doc_chunking")
print(f"{len(demo)} semantic chunks:\n")
for c in demo:
    print(f"[{len(c['text']):>3} chars] {c['text'][:80]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2 semantic chunks:

[447 chars] Chunking is the process of splitting documents into smaller pieces before indexi...
[265 chars] Overlap ensures that a sentence falling on a chunk boundary still appears intact...


### 3.5 Comparing the strategies & choosing one for the pipeline

**What this cell does:** runs all four chunkers over the whole corpus and tabulates chunk count and length statistics.

Read the table with the trade-off in mind:
- Fixed-size gives the most uniform lengths but the worst boundaries.
- Sentence/semantic give clean boundaries but variable lengths.

**Decision:** we proceed with **sentence-based chunks** for the rest of the notebook — clean boundaries, and small enough (≈2–3 sentences) that each embedding stays focused. Every chunk gets a global integer index, which is how FAISS and BM25 will refer to it.

In [ ]:
strategies = {
    "fixed":     lambda t, d: fixed_size_chunk(t, d),
    "recursive": lambda t, d: recursive_chunk(t, d),
    "sentence":  lambda t, d: sentence_chunk(t, d),
    "semantic":  lambda t, d: semantic_chunk(t, d),
}

rows = []
for name, fn in strategies.items():
    all_chunks = [c for doc_id, text in DOCUMENTS.items() for c in fn(text, doc_id)]
    lengths = [len(c["text"]) for c in all_chunks]
    rows.append({"strategy": name, "n_chunks": len(all_chunks),
                 "avg_len": int(np.mean(lengths)), "min_len": min(lengths), "max_len": max(lengths)})

display(pd.DataFrame(rows).set_index("strategy"))

# ---- Final corpus of chunks used by the rest of the notebook ----------------
CHUNKS = [c for doc_id, text in DOCUMENTS.items() for c in sentence_chunk(text, doc_id)]
CHUNK_TEXTS = [c["text"] for c in CHUNKS]
print(f"\nPipeline corpus: {len(CHUNKS)} sentence-based chunks")

,n_chunks,avg_len,min_len,max_len
strategy,,,,
fixed,19,304,26,400
recursive,19,261,43,396
sentence,16,405,268,543
semantic,16,310,62,596



Pipeline corpus: 16 sentence-based chunks


---
# 4. Embeddings — turning text into searchable vectors

**What this cell does:** embeds all chunks into a single `(n_chunks, 384)` matrix.

Key concepts packed into these few lines:

**Bi-encoder architecture.** `all-MiniLM-L6-v2` is a 6-layer transformer that maps any text to a 384-dim vector. Query and documents are encoded **independently** — that independence is what makes dense retrieval fast: document vectors are computed **once, offline**; at query time we embed only the query (one forward pass) and do cheap vector math against everything.

**`normalize_embeddings=True` — why it matters.** Cosine similarity = dot product ÷ (norms). If we scale every vector to unit length up front, the norms are 1, so:

$$\text{cosine}(q, d) = q \cdot d$$

A plain dot product (which FAISS's `IndexFlatIP` computes natively) then *is* cosine similarity. This is the standard trick — normalize once at index time, use fast inner-product search forever after.

**Model choice in practice.** MiniLM is the "small, fast, good enough" workhorse (~80 MB). Production upgrades: `bge-base-en-v1.5`, `e5-large`, or API embeddings (OpenAI `text-embedding-3`, Voyage, Cohere). The pipeline code is identical — only the vectors change. Check the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) and always benchmark on *your* data.

In [ ]:
# Embed every chunk → matrix of shape (n_chunks, 384)
chunk_embeddings = embedder.encode(
    CHUNK_TEXTS,
    normalize_embeddings=True,   # unit length → dot product == cosine similarity
    show_progress_bar=True,
)
print("Embedding matrix:", chunk_embeddings.shape)

# Sanity checks
print("Vector norm (should be 1.0):", round(float(np.linalg.norm(chunk_embeddings[0])), 4))

# Semantically related vs unrelated chunk similarity
sim_related   = float(np.dot(chunk_embeddings[0], chunk_embeddings[1]))   # both about chunking
sim_unrelated = float(np.dot(chunk_embeddings[0], chunk_embeddings[-1]))  # chunking vs evaluation
print(f"similarity(chunking, chunking) = {sim_related:.3f}")
print(f"similarity(chunking, eval)     = {sim_unrelated:.3f}   ← lower, as expected")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix: (16, 384)
Vector norm (should be 1.0): 1.0
similarity(chunking, chunking) = 0.646
similarity(chunking, eval)     = 0.317   ← lower, as expected


---
# 5. The vector index (FAISS) and dense retrieval

**What this cell does:** builds a FAISS index over the chunk embeddings and wraps search in a `dense_retrieve()` function used everywhere below.

**`IndexFlatIP` explained:**
- `Flat` = brute force: the query is compared against **every** stored vector. Exact, perfect recall.
- `IP` = inner product — which, because we normalized, equals cosine similarity.

**When brute force stops being fine.** Exact search is O(n) per query — perfectly fast up to a few hundred thousand vectors. Beyond that you trade a little recall for a lot of speed with **approximate nearest neighbour (ANN)** indexes:

| Index | Idea | Trade-off |
|---|---|---|
| `IndexIVFFlat` | Cluster vectors; search only the closest `nprobe` clusters | Tunable recall/speed; needs a training pass |
| `IndexHNSWFlat` | Navigable small-world graph, greedy hops toward the query | Sub-ms search, higher memory |
| + `PQ` variants | Compress vectors (product quantization) | 10–30× less RAM, slight accuracy loss |

Managed vector DBs (Pinecone, Weaviate, Qdrant, pgvector, OpenSearch) are these same structures plus persistence, filtering, and replication.

`dense_retrieve()` returns `(chunk_index, score)` pairs — the common currency all our retrievers will speak, so they can be fused and compared.

In [ ]:
import faiss

dim = chunk_embeddings.shape[1]           # 384
index = faiss.IndexFlatIP(dim)            # exact inner-product index
index.add(chunk_embeddings.astype(np.float32))
print(f"FAISS index holds {index.ntotal} vectors of dim {dim}")

def dense_retrieve(query, k=5):
    """Embed the query, return top-k (chunk_idx, cosine_score) pairs."""
    q = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, k)      # both are shape (1, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

# Demo
for idx, score in dense_retrieve("How large should my chunks be?", k=3):
    print(f"{score:.3f}  [{CHUNKS[idx]['doc_id']}]  {CHUNK_TEXTS[idx][:75]}...")

FAISS index holds 16 vectors of dim 384
0.522  [doc_chunking]  Chunking is the process of splitting documents into smaller pieces before i...
0.451  [doc_chunking]  A common starting point is 200 to 500 tokens per chunk with an overlap of 1...
0.081  [doc_vectordb]  For millions of vectors, approximate nearest neighbour indexes are used ins...


---
# 6. Lexical retrieval with BM25

**What this cell does:** builds a BM25 index over the same chunks and wraps it in `bm25_retrieve()` with the same `(chunk_idx, score)` interface.

**How BM25 scores.** For each query term, roughly:

$$\text{score} \mathrel{+}= IDF(t) \cdot \frac{tf(t, d) \cdot (k_1 + 1)}{tf(t, d) + k_1 \cdot (1 - b + b \cdot |d|/\text{avg}|d|)}$$

In words: rare terms (**IDF**) matter more; repeated terms help with **diminishing returns** (`k1` saturation); long documents are **penalised** (`b` length normalisation).

**Why keep BM25 in a world of embeddings?** Complementary failure modes:
- Embeddings blur rare exact tokens — "error `E4032`", "`RRF`", part numbers, people's names all map to fuzzy regions of vector space. BM25 nails them.
- BM25 is blind to synonyms — "made-up facts" won't match a document that says "hallucination". Embeddings handle that.

Our tokenizer is deliberately simple (lowercase + word regex). Production systems add stemming, stop-word handling, etc. — or use OpenSearch/Elasticsearch, which is BM25 at scale.

In [ ]:
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([simple_tokenize(t) for t in CHUNK_TEXTS])

def bm25_retrieve(query, k=5):
    """Return top-k (chunk_idx, bm25_score) pairs."""
    scores = bm25.get_scores(simple_tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top]

# Head-to-head: exact rare keyword vs paraphrase --------------------------------
for query in ["What is RRF?", "How can I catch made-up facts in answers?"]:
    print(f"\nQuery: {query!r}")
    d = dense_retrieve(query, k=1)[0]
    b = bm25_retrieve(query, k=1)[0]
    print(f"  dense → [{CHUNKS[d[0]]['doc_id']}]  bm25 → [{CHUNKS[b[0]]['doc_id']}]")


Query: 'What is RRF?'
  dense → [doc_hybrid]  bm25 → [doc_hybrid]

Query: 'How can I catch made-up facts in answers?'
  dense → [doc_halluc]  bm25 → [doc_vectordb]


**Reading the output above:** on the acronym query BM25 confidently lands on the RRF chunk, while on the paraphrase query ("made-up facts" → hallucination) dense retrieval wins and BM25 has nothing to grip. Neither retriever dominates — which is exactly the motivation for the next section.

---
# 7. Hybrid retrieval with Reciprocal Rank Fusion (RRF)

**What this cell does:** fuses the dense and BM25 ranked lists into one, using RRF.

**The score-scale problem.** Cosine scores live in roughly [0, 1]; BM25 scores are unbounded (0 to 15+ here). Naively adding them lets one retriever drown the other, and min-max normalisation is fragile (one outlier reshapes everything).

**RRF sidesteps scores entirely — it uses only ranks:**

$$RRF(d) = \sum_{r \,\in\, \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

- A document ranked #1 by both retrievers gets $\frac{1}{61} + \frac{1}{61}$ — agreement is rewarded.
- `k = 60` (the standard constant from the original paper) dampens the gap between rank 1 and rank 2, so a single retriever's top pick can't dominate a document that both retrievers *like*.
- No normalisation, no tuning, robust by construction — which is why RRF is the default fusion in most production hybrid systems (and built into OpenSearch/Elastic).

Note we ask each retriever for a **deeper list** (`depth=10`) than the final `k=5` — fusion needs overlap between the lists to detect agreement.

In [ ]:
def rrf_fuse(ranked_lists, k=60, top_k=5):
    """Reciprocal Rank Fusion over multiple [(chunk_idx, score), ...] ranked lists."""
    fused = {}
    for lst in ranked_lists:
        for rank, (idx, _score) in enumerate(lst, start=1):   # raw scores ignored — ranks only
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])[:top_k]

def hybrid_retrieve(query, k=5, depth=10):
    """Dense + BM25, fused with RRF. Fetch deeper lists than k so fusion has signal."""
    return rrf_fuse([dense_retrieve(query, depth), bm25_retrieve(query, depth)], top_k=k)

# Demo: hybrid handles BOTH query styles
for query in ["What is RRF?", "How can I catch made-up facts in answers?"]:
    top = hybrid_retrieve(query, k=1)[0]
    print(f"{query!r:55s} → [{CHUNKS[top[0]]['doc_id']}]")

'What is RRF?'                                          → [doc_hybrid]
'How can I catch made-up facts in answers?'             → [doc_halluc]


---
# 8. Retrieval optimisation: query expansion, HyDE, and MMR

First-stage retrieval fails most often not because the index is bad, but because the **query is a poor probe**: too short, wrong vocabulary, or ambiguous. Three standard fixes:

### 8.1 Multi-query expansion
Generate N paraphrases of the user's question with a small LLM, retrieve with **each** paraphrase, and RRF-fuse all the result lists. Each paraphrase probes a slightly different region of embedding space, so recall goes up. Cost: N extra retrievals + one LLM call.

**What this cell does:** loads `flan-t5-base` (a free, local, instruction-tuned model — also our generator later), prompts it for paraphrases, and fuses.

> In production you'd use a stronger LLM for paraphrasing; flan-t5's paraphrases are basic but demonstrate the mechanics.

In [ ]:
from transformers import pipeline as hf_pipeline

# Small local instruction-tuned LLM — no API key needed. (~1 GB download, one time.)
llm = hf_pipeline("text-generation", model="google/flan-t5-base",
                  device=0 if DEVICE == "cuda" else -1)

def generate_text(prompt, max_new_tokens=128):
    return llm(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"].strip()

def multi_query_retrieve(query, k=5, n_variants=2):
    """Paraphrase the query, retrieve with original + paraphrases, RRF-fuse everything."""
    variants = [query]
    for i in range(n_variants):
        v = generate_text(f"Rewrite this question using different words, keeping the meaning: {query}")
        if v and v.lower() != query.lower():
            variants.append(v)
    print("  query variants:", variants)
    all_lists = [hybrid_retrieve(v, k=10) for v in variants]
    return rrf_fuse(all_lists, top_k=k)

top = multi_query_retrieve("How do I stop my RAG bot from inventing things?", k=3)
for idx, s in top:
    print(f"  → [{CHUNKS[idx]['doc_id']}] {CHUNK_TEXTS[idx][:70]}...")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

  query variants: ['How do I stop my RAG bot from inventing things?', 'Rewrite this question using different words, keeping the meaning: How do I stop my RAG bot from inventing things?', 'Rewrite this question using different words, keeping the meaning: How do I stop my RAG bot from inventing things?']
  → [doc_bm25] BM25 requires no training and excels at exact keyword matches such as ...
  → [doc_halluc] Hallucination in retrieval-augmented generation means the model produc...
  → [doc_halluc] One automatic approach uses a natural language inference model: the an...


### 8.2 HyDE — Hypothetical Document Embeddings

**The asymmetry problem:** a *question* ("How does X work?") and its *answer passage* ("X works by...") are different kinds of text, and their embeddings sit in slightly different regions. Question→passage matching is inherently harder than passage→passage matching.

**HyDE's trick:** ask an LLM to *hallucinate a plausible answer* to the question, embed **that fake answer**, and search with it. The fake answer's *facts* may be wrong, but its *vocabulary, style, and structure* resemble a real answer passage — so it lands closer to the true passages in embedding space. We deliberately exploit hallucination for retrieval (while stamping it out in generation — Section 12).

**What this cell does:** implements HyDE in four lines — generate hypothetical answer → embed it → FAISS search. When to use it: zero-shot domains with short, terse user queries. When to skip: when queries are already long and descriptive, or the latency of an extra LLM call hurts.

In [ ]:
def hyde_retrieve(query, k=5):
    """Search with the embedding of a *hypothetical answer* instead of the raw question."""
    hypothetical = generate_text(
        f"Write a short, detailed paragraph that answers this question: {query}")
    print(f"  hypothetical doc: {hypothetical[:100]}...")
    q = embedder.encode([hypothetical], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

top = hyde_retrieve("Why use overlap when splitting documents?", k=3)
for idx, s in top:
    print(f"  {s:.3f} [{CHUNKS[idx]['doc_id']}] {CHUNK_TEXTS[idx][:70]}...")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  hypothetical doc: Write a short, detailed paragraph that answers this question: Why use overlap when splitting documen...
  0.564 [doc_chunking] Chunking is the process of splitting documents into smaller pieces bef...
  0.510 [doc_chunking] A common starting point is 200 to 500 tokens per chunk with an overlap...
  0.262 [doc_eval] Mean reciprocal rank, MRR, averages 1 divided by the rank of the first...


### 8.3 MMR — Maximal Marginal Relevance (diversity)

**The redundancy problem:** top-k similarity search often returns k *near-duplicates* — five chunks all saying the same thing — wasting the LLM's limited context window while a complementary fact sits at rank 6.

**MMR** re-selects results one at a time, balancing two forces:

$$MMR = \arg\max_{d \in \text{candidates}} \Big[ \lambda \cdot \text{sim}(d, q) \;-\; (1-\lambda) \cdot \max_{s \in \text{selected}} \text{sim}(d, s) \Big]$$

- First term: relevance to the query.
- Second term: penalty for resembling anything **already selected**.
- `λ = 1` → plain similarity ranking; `λ = 0.5` → strong diversity push.

**What this cell does:** implements MMR over a candidate pool of 10, selecting 3 with λ=0.6. Watch the output — MMR pulls in chunks from *different* documents where plain top-k would repeat one document.

In [ ]:
def mmr_retrieve(query, k=3, pool=10, lam=0.6):
    """Greedy MMR selection: relevant to the query, dissimilar to already-picked chunks."""
    q = embedder.encode([query], normalize_embeddings=True)[0]
    cand = [i for i, _ in dense_retrieve(query, k=pool)]     # candidate pool
    selected = []
    while cand and len(selected) < k:
        best, best_score = None, -1e9
        for c in cand:
            relevance = float(np.dot(chunk_embeddings[c], q))
            redundancy = max((float(np.dot(chunk_embeddings[c], chunk_embeddings[s]))
                              for s in selected), default=0.0)
            score = lam * relevance - (1 - lam) * redundancy
            if score > best_score:
                best, best_score = c, score
        selected.append(best)
        cand.remove(best)
    return selected

query = "How should I split and index documents for retrieval?"
print("Plain top-3 (note repeated docs):")
for idx, _ in dense_retrieve(query, k=3):
    print(f"  [{CHUNKS[idx]['doc_id']}]")
print("MMR top-3 (diverse docs):")
for idx in mmr_retrieve(query, k=3):
    print(f"  [{CHUNKS[idx]['doc_id']}]")

Plain top-3 (note repeated docs):
  [doc_chunking]
  [doc_hybrid]
  [doc_eval]
MMR top-3 (diverse docs):
  [doc_chunking]
  [doc_hybrid]
  [doc_rerank]


---
# 9. Reranking with a cross-encoder

**Bi-encoder vs cross-encoder — the core distinction of this section:**

| | Bi-encoder (Sections 4–5) | Cross-encoder (this section) |
|---|---|---|
| Input | Query and passage encoded **separately** | Query **and** passage in **one** forward pass |
| Attention | None between query and passage | Every query token attends to every passage token |
| Output | Two vectors → cosine | One direct relevance score |
| Speed | Precompute docs offline; O(1) model calls per query | One model call **per query–passage pair** |
| Accuracy | Good | Substantially better |

The cross-encoder is more accurate precisely because of that joint attention — it can see that "it" in the passage refers to the thing the query asked about. But it can't precompute anything, so running it over a whole corpus is impossible.

**The two-stage pattern (industry standard):**
1. **Recall stage** — cheap hybrid retrieval fetches ~20 candidates (goal: don't *miss* the answer).
2. **Precision stage** — the cross-encoder rescores those 20 and keeps the best 3–5 (goal: put the answer *first*, and feed the LLM only clean context).

**What this cell does:** loads `ms-marco-MiniLM-L-6-v2` (a cross-encoder fine-tuned on MS MARCO web-search relevance data) and implements `retrieve_and_rerank()`. The demo prints candidate order before and after — watch chunks jump.

In [ ]:
# Cross-encoder trained for relevance ranking (query, passage) → score
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

def retrieve_and_rerank(query, k=3, recall_k=20):
    """Stage 1: hybrid retrieval (recall). Stage 2: cross-encoder rerank (precision)."""
    candidates = [idx for idx, _ in hybrid_retrieve(query, k=recall_k)]
    pairs  = [(query, CHUNK_TEXTS[idx]) for idx in candidates]
    scores = reranker.predict(pairs)                       # one forward pass per pair
    order  = np.argsort(scores)[::-1][:k]
    return [(candidates[i], float(scores[i])) for i in order]

query = "Why are cross-encoders slower but more accurate than bi-encoders?"

print("Before rerank (hybrid order):")
for rank, (idx, s) in enumerate(hybrid_retrieve(query, k=5), 1):
    print(f"  {rank}. [{CHUNKS[idx]['doc_id']}]")

print("After cross-encoder rerank:")
for rank, (idx, s) in enumerate(retrieve_and_rerank(query, k=5, recall_k=10), 1):
    print(f"  {rank}. score={s:+.2f} [{CHUNKS[idx]['doc_id']}]")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Before rerank (hybrid order):
  1. [doc_rerank]
  2. [doc_rerank]
  3. [doc_embeddings]
  4. [doc_vectordb]
  5. [doc_embeddings]
After cross-encoder rerank:
  1. score=+7.05 [doc_rerank]
  2. score=+3.30 [doc_rerank]
  3. score=-1.26 [doc_embeddings]
  4. score=-9.75 [doc_hybrid]
  5. score=-10.39 [doc_vectordb]


---
# 10. Retrieval evaluation — which strategy actually wins?

Everything so far was qualitative. Now we measure, using the labelled queries from Section 2.

**The metrics:**

- **Hit@k (recall@k):** fraction of queries where a chunk from the correct document appears in the top k. Answers *"did the right context reach the LLM at all?"* — the ceiling on end-to-end quality.
- **MRR (Mean Reciprocal Rank):** average of $1/\text{rank}$ of the *first* correct chunk (0 if absent). Rank 1 → 1.0, rank 2 → 0.5, rank 3 → 0.33. Answers *"is the right context at the top?"* — which matters because LLMs attend more reliably to context that appears early ("lost in the middle" effect).

**What this cell does:** runs four strategies — dense-only, BM25-only, hybrid, hybrid+rerank — over every eval query and tabulates Hit@3 and MRR.

**How to read the result:** expect BM25 and dense to each miss *different* queries (the paraphrase vs acronym split from Section 6), hybrid to cover both, and reranking to push MRR up by promoting the right chunk to rank 1. On a 9-query set differences are coarse — in real projects you want 50–200 labelled queries before trusting a comparison.

In [ ]:
def evaluate_retriever(retrieve_fn, queries, k=3):
    """Compute Hit@k and MRR for a retriever over labelled queries."""
    hits, rr = 0, []
    for item in queries:
        results = retrieve_fn(item["q"], k)               # [(chunk_idx, score), ...]
        ranks = [CHUNKS[idx]["doc_id"] for idx, _ in results]
        if item["relevant_doc"] in ranks:
            hits += 1
            rr.append(1.0 / (ranks.index(item["relevant_doc"]) + 1))
        else:
            rr.append(0.0)
    return {"hit@3": hits / len(queries), "MRR": float(np.mean(rr))}

strategies = {
    "dense only":      lambda q, k: dense_retrieve(q, k),
    "bm25 only":       lambda q, k: bm25_retrieve(q, k),
    "hybrid (RRF)":    lambda q, k: hybrid_retrieve(q, k),
    "hybrid + rerank": lambda q, k: retrieve_and_rerank(q, k, recall_k=15),
}

results = {name: evaluate_retriever(fn, EVAL_QUERIES) for name, fn in strategies.items()}
display(pd.DataFrame(results).T.round(3))

,hit@3,MRR
dense only,1.0,1.0
bm25 only,1.0,1.0
hybrid (RRF),1.0,1.0
hybrid + rerank,1.0,1.0


---
# 11. Grounded generation

**What this cell does:** assembles the retrieved chunks into a **grounded prompt** and calls the LLM.

Anatomy of the prompt — each line exists for a reason:
1. **Role + constraint first:** *"Answer using ONLY the context"* — the single most important hallucination-prevention instruction.
2. **Numbered context blocks `[1] [2] [3]`:** enables citation and lets the faithfulness checker (next section) trace claims back to sources.
3. **The escape hatch:** *"If the context does not contain the answer, say 'I don't know'"* — without an explicit permission to abstain, LLMs fill gaps with plausible fabrications. This one sentence is the cheapest hallucination fix that exists.

**A note on the model:** `flan-t5-base` (250M params) produces short, sometimes clumsy answers — it's here so the notebook runs free and offline. The prompt structure is exactly what you'd send to Claude or GPT-4 via API; swap `generate_text()`'s internals and nothing else changes.

In [ ]:
def build_prompt(query, context_chunks):
    """Grounded prompt: numbered context + strict instruction + abstention escape hatch."""
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(context_chunks))
    return (
        "Answer the question using ONLY the context below. "
        "If the context does not contain the answer, say \"I don't know\".\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )

def rag_answer(query, k=3):
    """Full loop: retrieve+rerank → build grounded prompt → generate."""
    top = retrieve_and_rerank(query, k=k)
    context_chunks = [CHUNK_TEXTS[idx] for idx, _ in top]
    answer = generate_text(build_prompt(query, context_chunks), max_new_tokens=150)
    return answer, context_chunks

query = "Why does chunk overlap help retrieval?"
answer, ctx = rag_answer(query)
print("Q:", query)
print("A:", answer)
print("\nContext used:")
for i, c in enumerate(ctx, 1):
    print(f"  [{i}] {c[:80]}...")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Why does chunk overlap help retrieval?
A: Answer the question using ONLY the context below. If the context does not contain the answer, say "I don't know".

Context:
[1] Chunking is the process of splitting documents into smaller pieces before indexing them in a retrieval system. Chunk size involves a fundamental trade-off: small chunks produce precise, focused embeddings but lose surrounding context, while large chunks preserve context but dilute the embedding signal, because a single vector must summarise many topics. A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent.

[2] A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent. Overlap ensures that a sentence falling on a chunk boundary still appears intact in at least one chunk. Recursive character splitting tries to break text at natural boundaries, first at paragraphs, then sentences, then words, only cutting mid-word as a last resort.

[3] Hybrid retr

---
# 12. Hallucination evaluation

Even with a grounded prompt, models drift beyond their context. We measure two complementary things:

| Metric | Question it answers | Failure it catches |
|---|---|---|
| **Faithfulness** | Is every claim in the answer *supported by the retrieved context*? | Fabrication / hallucination |
| **Answer relevance** | Does the answer actually *address the question*? | Faithful-but-useless answers (e.g. "I don't know", or on-context but off-question) |

An answer can score high on one and low on the other — you need both. (These two, plus context precision/recall, are the core of the **RAGAS** framework; here we build them by hand so the mechanics are visible.)

### 12.1 NLI-based faithfulness

**The idea:** Natural Language Inference models classify a (premise, hypothesis) pair as **entailment / neutral / contradiction**. We treat the retrieved **context as the premise** and each **answer sentence as a hypothesis**:

```
faithfulness = (# answer sentences entailed by the context) / (# answer sentences)
```

- 1.0 → every claim is backed by the context (fully grounded).
- A sentence classified *neutral* is a claim the context doesn't support — the classic subtle hallucination.
- *Contradiction* is worse: the answer disagrees with its own sources.

**What this cell does:** loads a small NLI cross-encoder (`nli-deberta-v3-small`), splits the answer into sentences, scores each (context, sentence) pair, and reports the per-sentence verdicts plus the aggregate score. This runs locally, deterministically, and per-sentence — properties that make NLI checkers popular as automated guardrails.

In [ ]:
# NLI cross-encoder: (premise, hypothesis) → logits for [contradiction, entailment, neutral]
nli = CrossEncoder("cross-encoder/nli-deberta-v3-small", device=DEVICE)
NLI_LABELS = ["contradiction", "entailment", "neutral"]

def faithfulness_score(answer, context_chunks, verbose=True):
    """Fraction of answer sentences entailed by the concatenated context."""
    premise = " ".join(context_chunks)
    sentences = [s for s in sent_tokenize(answer) if len(s.split()) >= 3]
    if not sentences:
        return 0.0
    logits = nli.predict([(premise, s) for s in sentences])
    supported = 0
    for s, lg in zip(sentences, logits):
        label = NLI_LABELS[int(np.argmax(lg))]
        supported += (label == "entailment")
        if verbose:
            print(f"  [{label:^13}] {s[:80]}")
    return supported / len(sentences)

# Test 1: the real RAG answer from Section 11
print("Grounded answer:")
score = faithfulness_score(answer, ctx)
print(f"→ faithfulness = {score:.2f}\n")

# Test 2: a deliberately hallucinated answer against the SAME context
fake = ("Chunk overlap was invented by Google in 2015. "
        "It guarantees 100 percent retrieval accuracy in all cases. "
        "Overlap ensures a sentence on a chunk boundary appears intact in at least one chunk.")
print("Hallucinated answer:")
score = faithfulness_score(fake, ctx)
print(f"→ faithfulness = {score:.2f}   (only the last sentence is supported)")

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Grounded answer:
  [contradiction] Answer the question using ONLY the context below.
  [   neutral   ] If the context does not contain the answer, say "I don't know".
  [   neutral   ] Context:
[1] Chunking is the process of splitting documents into smaller pieces 
  [   neutral   ] Chunk size involves a fundamental trade-off: small chunks produce precise, focus
  [   neutral   ] A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 
  [   neutral   ] [2] A common starting point is 200 to 500 tokens per chunk with an overlap of 10
  [   neutral   ] Overlap ensures that a sentence falling on a chunk boundary still appears intact
  [   neutral   ] Recursive character splitting tries to break text at natural boundaries, first a
  [   neutral   ] [3] Hybrid retrieval combines dense vector search with lexical BM25 search to ge
  [   neutral   ] A popular fusion method is Reciprocal Rank Fusion, abbreviated RRF, which scores
  [   neutral   ] RRF needs no score norm

### 12.2 Answer relevance

**What this cell does:** measures whether the answer addresses the *question*, via cosine similarity between the question embedding and the answer embedding.

(The full RAGAS version is fancier — it asks an LLM to *reverse-generate* questions from the answer and compares those to the original question — but embedding similarity captures the essence and costs nothing.)

**Why faithfulness alone isn't enough:** the answer *"BM25 is a lexical ranking function"* is perfectly faithful to our context — but if the question was about chunk overlap, it's useless. Relevance catches that; the demo below shows exactly this case.

In [ ]:
def answer_relevance(query, answer):
    """Cosine similarity between question and answer embeddings."""
    q, a = embedder.encode([query, answer], normalize_embeddings=True)
    return float(np.dot(q, a))

print(f"relevant answer  : {answer_relevance(query, answer):.3f}")
print(f"off-topic answer : {answer_relevance(query, 'BM25 is a lexical ranking function based on term frequency.'):.3f}")

relevant answer  : 0.689
off-topic answer : 0.218


### 12.3 LLM-as-judge (the production pattern)

**What this cell does:** shows the third approach — prompt an LLM to act as a **claim verifier**: extract claims from the answer, check each against the context, output a verdict.

Trade-offs vs the NLI approach:

| | NLI cross-encoder | LLM-as-judge |
|---|---|---|
| Cost | Tiny, local, free | API call per evaluation |
| Nuance | Sentence-level entailment only | Handles multi-hop reasoning, partial support, numeric checks |
| Determinism | Fully deterministic | Mostly (temperature 0) |
| Judge quality | Fixed | Scales with judge model strength |

In production the common recipe is: **NLI as a cheap always-on guardrail** on every response, **LLM-as-judge in offline batch evaluation** (with a strong model) when comparing pipeline versions. Our local flan-t5 is too weak to be a trustworthy judge — the cell demonstrates the prompt pattern you'd send to a strong model.

In [ ]:
JUDGE_PROMPT = """You are a strict fact-checking judge.

Context:
{context}

Answer to verify:
{answer}

Is every claim in the answer fully supported by the context? Reply with exactly one word:
SUPPORTED or UNSUPPORTED."""

def llm_judge(answer, context_chunks):
    prompt = JUDGE_PROMPT.format(context=" ".join(context_chunks), answer=answer)
    return generate_text(prompt, max_new_tokens=8)

print("real answer  →", llm_judge(answer, ctx))
print("fake answer  →", llm_judge(fake, ctx))
# Note: flan-t5-base is a weak judge — treat this as a demo of the PROMPT PATTERN.
# In production, send this prompt to a strong model (Claude / GPT-4) at temperature 0.

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (687 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


real answer  → You are a strict fact-checking judge.

Context:
Chunking is the process of splitting documents into smaller pieces before indexing them in a retrieval system. Chunk size involves a fundamental trade-off: small chunks produce precise, focused embeddings but lose surrounding context, while large chunks preserve context but dilute the embedding signal, because a single vector must summarise many topics. A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent. A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent. Overlap ensures that a sentence falling on a chunk boundary still appears intact in at least one chunk. Recursive character splitting tries to break text at natural boundaries, first at paragraphs, then sentences, then words, only cutting mid-word as a last resort. Hybrid retrieval combines dense vector search with lexical BM25 search to get the best of both worlds. A popular fusion method is R

[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


fake answer  → You are a strict fact-checking judge.

Context:
Chunking is the process of splitting documents into smaller pieces before indexing them in a retrieval system. Chunk size involves a fundamental trade-off: small chunks produce precise, focused embeddings but lose surrounding context, while large chunks preserve context but dilute the embedding signal, because a single vector must summarise many topics. A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent. A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent. Overlap ensures that a sentence falling on a chunk boundary still appears intact in at least one chunk. Recursive character splitting tries to break text at natural boundaries, first at paragraphs, then sentences, then words, only cutting mid-word as a last resort. Hybrid retrieval combines dense vector search with lexical BM25 search to get the best of both worlds. A popular fusion method is R

---
# 13. The complete end-to-end pipeline

**What this cell does:** consolidates everything into one `RAGPipeline` class — the shape this code takes when it leaves a notebook and becomes a service.

Follow `query()` top to bottom; it is the whole notebook in eight lines:
1. *(optional)* expand the query (Section 8),
2. hybrid retrieve a deep candidate list (Sections 5–7),
3. cross-encoder rerank to a short, precise context (Section 9),
4. grounded generation (Section 11),
5. score faithfulness + relevance and attach them to the response (Section 12).

Returning the **evaluation scores with every answer** is the production-grade habit: log them, alert when faithfulness drops below a threshold (e.g. 0.7 → show a "low confidence" banner or trigger a retry with more context), and you have observable, self-monitoring RAG instead of a black box.

In [ ]:
class RAGPipeline:
    """End-to-end RAG: hybrid retrieval → rerank → grounded generation → self-evaluation."""

    def __init__(self, chunks, chunk_texts, faiss_index, bm25_index,
                 embedder, reranker, nli, generate_fn):
        self.chunks, self.texts = chunks, chunk_texts
        self.index, self.bm25 = faiss_index, bm25_index
        self.embedder, self.reranker, self.nli = embedder, reranker, nli
        self.generate = generate_fn

    # ---- retrieval stages ----------------------------------------------------
    def _dense(self, q, k):  return dense_retrieve(q, k)
    def _bm25(self, q, k):   return bm25_retrieve(q, k)

    def _retrieve(self, query, k, recall_k):
        candidates = rrf_fuse([self._dense(query, recall_k), self._bm25(query, recall_k)],
                              top_k=recall_k)
        pairs  = [(query, self.texts[i]) for i, _ in candidates]
        scores = self.reranker.predict(pairs)
        order  = np.argsort(scores)[::-1][:k]
        return [candidates[i][0] for i in order]

    # ---- full query path -------------------------------------------------------
    def query(self, question, k=3, recall_k=15):
        ids     = self._retrieve(question, k, recall_k)
        context = [self.texts[i] for i in ids]
        answer  = self.generate(build_prompt(question, context), 150)
        return {
            "question":    question,
            "answer":      answer,
            "sources":     [self.chunks[i]["doc_id"] for i in ids],
            "faithfulness": faithfulness_score(answer, context, verbose=False),
            "relevance":    answer_relevance(question, answer),
        }

rag = RAGPipeline(CHUNKS, CHUNK_TEXTS, index, bm25, embedder, reranker, nli, generate_text)

for q in ["What is the trade-off between small and large chunks?",
          "How does reciprocal rank fusion work?",
          "How do I detect hallucinations in RAG answers?"]:
    r = rag.query(q)
    print(f"Q: {r['question']}")
    print(f"A: {r['answer']}")
    print(f"   sources={r['sources']}  faithfulness={r['faithfulness']:.2f}  relevance={r['relevance']:.2f}\n")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the trade-off between small and large chunks?
A: Answer the question using ONLY the context below. If the context does not contain the answer, say "I don't know".

Context:
[1] Chunking is the process of splitting documents into smaller pieces before indexing them in a retrieval system. Chunk size involves a fundamental trade-off: small chunks produce precise, focused embeddings but lose surrounding context, while large chunks preserve context but dilute the embedding signal, because a single vector must summarise many topics. A common starting point is 200 to 500 tokens per chunk with an overlap of 10 to 20 percent.

[2] For millions of vectors, approximate nearest neighbour indexes are used instead: IVF partitions the space into clusters and searches only the closest clusters, while HNSW builds a navigable small-world graph that achieves sub-millisecond search with a small recall loss. The choice is a recall versus latency versus memory trade-off.

[3] A common starting po

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How does reciprocal rank fusion work?
A: Answer the question using ONLY the context below. If the context does not contain the answer, say "I don't know".

Context:
[1] Hybrid retrieval combines dense vector search with lexical BM25 search to get the best of both worlds. A popular fusion method is Reciprocal Rank Fusion, abbreviated RRF, which scores each document as the sum of 1 / (k + rank) across the ranked lists it appears in, with k typically set to 60. RRF needs no score normalisation because it uses only ranks, not raw scores, which makes it robust when the two retrievers produce scores on completely different scales.

[2] Mean reciprocal rank, MRR, averages 1 divided by the rank of the first relevant document, so it rewards putting the right document at position one. A solid evaluation set contains queries with known relevant documents, including paraphrased queries that avoid the document's own vocabulary, which specifically stress-tests semantic matching.

[3] Retrieval qu

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I detect hallucinations in RAG answers?
A: Answer the question using ONLY the context below. If the context does not contain the answer, say "I don't know".

Context:
[1] Hallucination in retrieval-augmented generation means the model produces statements that are not supported by the retrieved context. Faithfulness evaluation checks whether every claim in the generated answer is entailed by the retrieved passages. One automatic approach uses a natural language inference model: the answer is split into sentences, and each sentence is checked for entailment against the context, yielding a faithfulness score equal to the fraction of supported sentences.

[2] One automatic approach uses a natural language inference model: the answer is split into sentences, and each sentence is checked for entailment against the context, yielding a faithfulness score equal to the fraction of supported sentences. Another approach is LLM-as-judge, where a strong model is prompted to verify each cla

---
# 14. Wrap-up: what you built, and what changes in production

## The mental model

```
Chunking      →  determines WHAT can be retrieved (unit of retrieval)
Embeddings    →  determine what "similar" MEANS
Hybrid + RRF  →  covers both semantic AND exact-keyword queries
Query opt     →  fixes the QUERY (expansion, HyDE) and the RESULT SET (MMR)
Reranking     →  recall first (cheap, wide), precision second (expensive, narrow)
Hit@k / MRR   →  proves which retrieval choices are better — never tune blind
Faithfulness  →  is the answer grounded in the context?
Relevance     →  does the answer address the question?
```

## Notebook → production checklist

| Notebook shortcut | Production replacement |
|---|---|
| 8 hand-written docs | Real ingestion: parsers (PDF/HTML), metadata, incremental re-indexing |
| `IndexFlatIP` (exact) | HNSW/IVF in a vector DB (Qdrant, pgvector, OpenSearch, Pinecone) with filtering |
| MiniLM embeddings | Benchmark stronger models (bge, e5, API embeddings) **on your own labelled queries** |
| flan-t5-base generator | Claude / GPT via API; add citation formatting and streaming |
| flan-t5 judge | Strong LLM judge offline; NLI guardrail online |
| 9 eval queries | 50–200 labelled queries; run the Section-10 harness in CI on every pipeline change |
| — | Caching, latency budgets (rerank depth is your main lever), observability on the per-answer scores |

## Ideas to extend this notebook
1. **Chunking A/B test** — re-run the Section 10 harness with fixed vs recursive vs semantic chunks and see which wins on MRR.
2. **Parent-document retrieval** — retrieve small chunks, but feed the LLM their *parent* section (precision of small chunks + context of large ones).
3. **RAGAS** — `pip install ragas` and compare its faithfulness/relevance scores to our hand-built ones.
4. **Metadata filtering** — add `doc_type`/`date` metadata and filter before vector search.
5. **Swap in an API LLM** — replace `generate_text()` with an Anthropic/OpenAI call and watch answer quality (and judge quality) jump.
